# SAMOSA Tutorial: Coupled sampling from two Gaussian distributions

This notebook demonstrates *coupled* sampling strategies. We will use a coupled Metropolis Hastings kernel with different coupled proposals and a shared accept or reject step. Each section runs a different coupled proposals and then shows scatter, trace and lag plots.

In [ ]:
# Some imports
import numpy as np
from samosa.utils.tools import lognormpdf
from samosa.utils.post_processing import (
    load_coupled_samples,
    get_position_from_states,
    scatter_matrix,
    plot_trace,
    plot_lag,
    joint_plots,
)
from typing import Any, Dict
import matplotlib.pyplot as plt

burnin = 0.25

## 1. Model Definition

We define models similar to how we defined them in the single chain case. Take a look at the `single_chain` notebook for more details.

In [ ]:
# Define the two (2D) Gaussian models - coarse and fine
# The coarse target is a Gaussian with mean [3, 4] and covariance [[2, 0], [0, 1]]
# The fine target is a Gaussian with mean [2, 3] and covariance [[2, 0.5], [0.5, 1]]
# These are the first two Gaussians in the family of Gaussians with
# mean [2^{-l+2}, 3^{-l+2}] and covariance [[2, 2^{-l}], [2^{-l}, 1]]

mean_coarse = np.array([[3], [4]])
cov_coarse = np.array([[2, 0], [0, 1]])

mean_fine = np.array([[2], [3]])
cov_fine = np.array([[2, 0], [0, 1]])


def gaussian_model_coarse(params: np.ndarray) -> Dict[str, Any]:
    """
    Coarse Gaussian model: log posterior via lognormpdf
    """
    lp = lognormpdf(params, mean=mean_coarse, cov=cov_coarse)
    output = {
        "log_posterior": lp,
        "cost": 1,
    }
    return output


def gaussian_model_fine(params: np.ndarray) -> Dict[str, Any]:
    """
    Fine Gaussian model: log posterior via lognormpdf
    """
    lp = lognormpdf(params, mean=mean_fine, cov=cov_fine)
    output = {
        "log_posterior": lp,
        "cost": 1,
    }
    return output


# Plot the two Gaussian contours to see what you're sampling from
# Plot the banana_model contours to see what you're sampling from
xlims = [-2, 8]
ylims = [-2, 10]
x = np.linspace(xlims[0], xlims[1], 100)
y = np.linspace(ylims[0], ylims[1], 100)
X, Y = np.meshgrid(x, y)
points = np.vstack([X.ravel(), Y.ravel()])  # (2, N)
Z1 = np.exp(gaussian_model_coarse(points)["log_posterior"]).reshape(X.shape)
Z2 = np.exp(gaussian_model_fine(points)["log_posterior"]).reshape(X.shape)
plt.contour(X, Y, Z1, levels=10, cmap="viridis")
plt.contour(X, Y, Z2, levels=10, cmap="plasma")
plt.show()

## Overview

Similar to the single chain case, we first build a *coupled* proposal, use that proposal to define a *coupled* kernel and use that kernel inside a *coupled* sampler.

Every coupled proposal in `SAMOSA` requires atleast a coarse proposal and a fine proposal. These individual proposals can be adapted or use transport maps.

We will explore three different coupled proposals, 1) Independent coupling, 2) Maximal coupling and 3) Synce coupling.

After sampling the coupled chains, we plot scatter, trace and lag plots as usual. In addition to these plots, we also plot joint plots, a dimension-wise scatter plot between the coarse and fine samples. This plot gives us a *visual* feel for how **correlated** the samples are.


## 1. Independent coupled proposal
`
Madrigal-Cianci, Juan P., Fabio Nobile, and Raúl Tempone. "Analysis of a class of multilevel Markov chain Monte Carlo algorithms based on independent Metropolis–Hastings." SIAM/ASA Journal on Uncertainty Quantification 11, no. 1 (2023): 91-138.
`

Uses an independent sampler to propose the same point for both the coupled chains. 

In [ ]:
from samosa import IndependentProposal, IndependentCoupling
from samosa import AdaptiveProposal, HaarioAdapter
from samosa import CoupledKernel
from samosa import CoupledChainSampler

# Define coarse and fine proposals. IndependentCoupling builds
# the shared independent sampler from these proposal parameters.
proposal_coarse = IndependentProposal(mu=mean_coarse, cov=cov_coarse)
proposal_fine = IndependentProposal(mu=mean_fine, cov=cov_fine)
# Uncomment to use adaptive proposals
proposal_coarse = AdaptiveProposal(
    proposal_coarse, HaarioAdapter(scale=2.38**2 / 2, adapt_start=1000, adapt_end=10000)
)
proposal_fine = AdaptiveProposal(
    proposal_fine, HaarioAdapter(scale=2.38**2 / 2, adapt_start=1000, adapt_end=10000)
)
proposal = IndependentCoupling(proposal_coarse, proposal_fine)
kernel = CoupledKernel(
    coarse_model=gaussian_model_coarse,
    fine_model=gaussian_model_fine,
    coupled_proposal=proposal,
)
sampler = CoupledChainSampler(
    kernel,
    initial_position_coarse=mean_coarse,
    initial_position_fine=mean_fine,
    n_iterations=10000,
    print_iteration=1000,
)
output = "IndependentCouplingGaussian"
ar1, ar2 = sampler.run(output)
print("Acceptance rate:", ar1)
print("Acceptance rate:", ar2)

# Plots
samples_coarse, samples_fine = load_coupled_samples(output)
positions_coarse = get_position_from_states(samples_coarse, burnin)
positions_fine = get_position_from_states(samples_fine, burnin)
fig, _, _ = scatter_matrix(
    [positions_coarse, positions_fine], sample_labels=["coarse", "fine"]
)
plt.show()
fig, _ = plot_trace(
    [positions_coarse, positions_fine], sample_labels=["coarse", "fine"]
)
plt.show()
fig, *_ = plot_lag([positions_coarse, positions_fine], sample_labels=["coarse", "fine"])
plt.show()
figs = joint_plots([positions_coarse, positions_fine])
plt.show()

## 2. Maximal coupling

`Thorisson, Hermann. "Coupling methods in probability theory." Scandinavian journal of statistics (1995): 159-182.`

A very nice algorithm for maximally coupling two proposals. Maximizes the probability that the proposals propose the same state. Can be related to the total variation distance between the two proposals.

In [ ]:
from samosa import GaussianRandomWalk, MaximalCoupling
from samosa import AdaptiveProposal, HaarioAdapter
from samosa import CoupledKernel
from samosa import CoupledChainSampler

# Define coarse and fine proposals. IndependentCoupling builds
# the shared independent sampler from these proposal parameters.
# For GaussianRandomWalk, use zero-mean increments; nonzero means add drift.
proposal_coarse = GaussianRandomWalk(mu=np.zeros_like(mean_coarse), cov=cov_coarse)
proposal_fine = GaussianRandomWalk(mu=np.zeros_like(mean_fine), cov=cov_fine)
# Uncomment to use adaptive proposals
# proposal_coarse = AdaptiveProposal(proposal_coarse, HaarioAdapter(scale=2.38**2/2, adapt_start=1000, adapt_end=10000))
# proposal_fine = AdaptiveProposal(proposal_fine, HaarioAdapter(scale=2.38**2/2, adapt_start=1000, adapt_end=10000))
proposal = MaximalCoupling(proposal_coarse, proposal_fine)
kernel = CoupledKernel(
    coarse_model=gaussian_model_coarse,
    fine_model=gaussian_model_fine,
    coupled_proposal=proposal,
)
sampler = CoupledChainSampler(
    kernel,
    initial_position_coarse=mean_coarse,
    initial_position_fine=mean_fine,
    n_iterations=10000,
    print_iteration=1000,
)
output = "MaximalCouplingGaussian"
ar1, ar2 = sampler.run(output)
print("Acceptance rate:", ar1)
print("Acceptance rate:", ar2)

# Plots
samples_coarse, samples_fine = load_coupled_samples(output)
positions_coarse = get_position_from_states(samples_coarse, burnin)
positions_fine = get_position_from_states(samples_fine, burnin)
fig, _, _ = scatter_matrix(
    [positions_coarse, positions_fine], sample_labels=["coarse", "fine"]
)
plt.show()
fig, _ = plot_trace(
    [positions_coarse, positions_fine], sample_labels=["coarse", "fine"]
)
plt.show()
fig, *_ = plot_lag([positions_coarse, positions_fine], sample_labels=["coarse", "fine"])
plt.show()
figs = joint_plots([positions_coarse, positions_fine])
plt.show()

## 3. SYNCE coupling

`Muchandimath, Sanjan, and Alex Gorodetsky. "Synchronized step Multilevel Markov chain Monte Carlo." arXiv preprint arXiv:2501.16538 (2025).`

Common random number coupling. Increases correlation between sample required for multi-fidelity inference

In [ ]:
from samosa import GaussianRandomWalk, SynceCoupling
from samosa import AdaptiveProposal, HaarioAdapter
from samosa import CoupledKernel
from samosa import CoupledChainSampler

# Define coarse and fine proposals.
proposal_coarse = GaussianRandomWalk(mu=np.zeros_like(mean_coarse), cov=cov_coarse)
proposal_fine = GaussianRandomWalk(mu=np.zeros_like(mean_fine), cov=cov_fine)
# Uncomment to use adaptive proposals
# proposal_coarse = AdaptiveProposal(proposal_coarse, HaarioAdapter(scale=2.38**2/2, adapt_start=1000, adapt_end=10000))
# proposal_fine = AdaptiveProposal(proposal_fine, HaarioAdapter(scale=2.38**2/2, adapt_start=1000, adapt_end=10000))
# Get some samples to build an initial transport map

proposal = SynceCoupling(proposal_coarse, proposal_fine)
kernel = CoupledKernel(
    coarse_model=gaussian_model_coarse,
    fine_model=gaussian_model_fine,
    coupled_proposal=proposal,
)
sampler = CoupledChainSampler(
    kernel,
    initial_position_coarse=mean_coarse,
    initial_position_fine=mean_fine,
    n_iterations=10000,
    print_iteration=1000,
)
output = "SynceCouplingGaussian"
ar1, ar2 = sampler.run(output)
print("Acceptance rate:", ar1)
print("Acceptance rate:", ar2)

# Plots
samples_coarse, samples_fine = load_coupled_samples(output)
positions_coarse = get_position_from_states(samples_coarse, burnin)
positions_fine = get_position_from_states(samples_fine, burnin)
fig, _, _ = scatter_matrix(
    [positions_coarse, positions_fine], sample_labels=["coarse", "fine"]
)
plt.show()
fig, _ = plot_trace(
    [positions_coarse, positions_fine], sample_labels=["coarse", "fine"]
)
plt.show()
fig, *_ = plot_lag([positions_coarse, positions_fine], sample_labels=["coarse", "fine"])
plt.show()
figs = joint_plots([positions_coarse, positions_fine])
plt.show()

## 4. Transport map + Coupling in the reference space

Now we will use transport maps to to map points of coupled chains to a common reference space (either the coarse model or a standard Gaussian) and use coupling techniques in this shared reference space. Using transport maps enables us to explore complex posterior shapes to improve sampling efficiency while using coupling techniques in the reference space will improve correlations.

In [ ]:
from samosa import GaussianRandomWalk, TransportProposal, LowerTriangularMap
from samosa import AdaptiveProposal, HaarioAdapter
from samosa import SynceCoupling
from samosa import CoupledKernel
from samosa import CoupledChainSampler

# Define coarse and fine proposals. IndependentCoupling builds
# the shared independent sampler from these proposal parameters.
# The base proposal will be used in the reference space so provide covariance accordingly
proposal_coarse = GaussianRandomWalk(mu=np.zeros_like(mean_coarse), cov=np.eye(2))
proposal_fine = GaussianRandomWalk(mu=np.zeros_like(mean_fine), cov=np.eye(2))
# Uncomment to use adaptive proposals (Also remove the scaling in the base proposal)
proposal_coarse = AdaptiveProposal(
    proposal_coarse, HaarioAdapter(scale=2.38**2 / 2, adapt_start=1000, adapt_end=10000)
)
proposal_fine = AdaptiveProposal(
    proposal_fine, HaarioAdapter(scale=2.38**2 / 2, adapt_start=1000, adapt_end=10000)
)
# Get some samples to build an initial transport map
samples_coarse, samples_fine = load_coupled_samples(
    "SynceCouplingGaussian"
)  # Make sure you run the previous example first
n_adapt = min(2000, len(samples_coarse), len(samples_fine))
tmap_coarse = LowerTriangularMap(
    dim=2,
    total_order=2,
    adapt_start=1000,
    adapt_end=10000,
    adapt_interval=500,
    initial_samples=samples_coarse[:n_adapt],
)
tmap_fine = LowerTriangularMap(
    dim=2,
    total_order=2,
    adapt_start=1000,
    adapt_end=10000,
    adapt_interval=500,
    initial_samples=samples_fine[:n_adapt],
)
# Wrap the base proposal with the transport map. The base proposal could be adapted as well, adaptation happens in the reference space
proposal_coarse = TransportProposal(proposal_coarse, tmap_coarse)
proposal_fine = TransportProposal(proposal_fine, tmap_fine)
# Wrapping proposals with transport maps and then supplying them to the coupling proposal will couple the chains in the reference space
proposal = SynceCoupling(proposal_coarse, proposal_fine)
# Everything else is the same as the previous examples
kernel = CoupledKernel(
    coarse_model=gaussian_model_coarse,
    fine_model=gaussian_model_fine,
    coupled_proposal=proposal,
)
sampler = CoupledChainSampler(
    kernel,
    initial_position_coarse=np.random.randn(2).reshape(2, 1),
    initial_position_fine=np.random.randn(2).reshape(2, 1),
    n_iterations=10000,
    print_iteration=1000,
)
output = "TransportMapSynceCouplingGaussian"
ar1, ar2 = sampler.run(output)
print("Acceptance rate:", ar1)
print("Acceptance rate:", ar2)

# Plots
samples_coarse, samples_fine = load_coupled_samples(output)
positions_coarse = get_position_from_states(samples_coarse, burnin)
positions_fine = get_position_from_states(samples_fine, burnin)
fig, _, _ = scatter_matrix(
    [positions_coarse, positions_fine], sample_labels=["coarse", "fine"]
)
plt.show()
fig, _ = plot_trace(
    [positions_coarse, positions_fine], sample_labels=["coarse", "fine"]
)
plt.show()
fig, *_ = plot_lag([positions_coarse, positions_fine], sample_labels=["coarse", "fine"])
plt.show()
figs = joint_plots([positions_coarse, positions_fine])
plt.show()

## 4.1 Transport map and reference point diagnostics

We can visually check the quality of the transport map for each individual posterior and also plot scatter, trace, lag and joint plots for the reference space points!

In [ ]:
from samosa import get_reference_position_from_states

refrenced_positions_coarse = get_reference_position_from_states(samples_coarse, burnin)
refrenced_positions_fine = get_reference_position_from_states(samples_fine, burnin)
fig, _, _ = scatter_matrix(
    [refrenced_positions_coarse, refrenced_positions_fine],
    sample_labels=["coarse", "fine"],
)
plt.show()
fig, _ = plot_trace(
    [refrenced_positions_coarse, refrenced_positions_fine],
    sample_labels=["coarse", "fine"],
)
plt.show()
fig, *_ = plot_lag(
    [refrenced_positions_coarse, refrenced_positions_fine],
    sample_labels=["coarse", "fine"],
)
plt.show()
figs = joint_plots([refrenced_positions_coarse, refrenced_positions_fine])
plt.show()